
# V45 LIVE — Full-Data Refit of Proven V38 Components

**Protected public-LB champion:** V38 = **0.608**  
**Minimum target:** **0.6151**  
**Main target:** **0.620**

V42 sarcasm, V43 identity attack, and V44 sexual-content specialists did not beat their protected baselines strongly enough.

V45 therefore changes strategy.

## Core idea

The competition permits combining labeled training and validation data for the final system.  
Our accepted LIVE partition contains:

- train: 3297
- validation: 500
- combined labeled data: **3797**
- test: 1000

Many V38 base models were 5-fold models, meaning each individual fold model saw only about 80% of the 3797 labeled rows. V45 refits the **already-proven V38 model families on all 3797 labeled rows**.

We only revisit the three components that public-leaderboard ablations proved were useful:

1. `is_toxic`
2. `implicit_toxicity`
3. `threatening`

Everything else stays **exactly V38**.

## V45 models

### Toxic full refit
Fine-tune `aplycaebous/tb-XLM-R-fpt` on all 3797 labeled examples with the same multilingual/multilabel setup used by the successful transliteration-aware toxic component.

### Threat full refit
Encode all labeled/test text with `nahidstaq/bge-m3-bangla`, then train a full-data logistic threat classifier.

### Implicit full meta refit
Refit the successful V36 regularized meta stacker on all 3797 OOF rows rather than only the 3297 training rows.

## Safety

V45 does **not automatically recommend a Kaggle submission**.

It creates:
- toxic-only refit candidate
- implicit-only refit candidate
- threat-only refit candidate
- combined refit candidate

All preserve the V38 positive count for the changed label.

Upload `v45_results.zip` to ChatGPT before submitting.


In [ ]:

# 1. Upload bundle
from google.colab import files
import pathlib,zipfile

uploaded=files.upload()
assert "v45_LIVE_full_refit_data.zip" in uploaded

DATA_DIR=pathlib.Path("/content/v45_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v45_LIVE_full_refit_data.zip") as z:
    z.extractall(DATA_DIR)

for p in sorted(DATA_DIR.iterdir()):
    print(p.name,f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Install dependencies + model preflight
import sys,subprocess

subprocess.check_call([
    sys.executable,"-m","pip","install","-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "sentence-transformers>=3.0.0",
    "scikit-learn>=1.4",
    "scipy",
    "safetensors",
    "tqdm",
])

from transformers import AutoConfig,AutoTokenizer
from sentence_transformers import SentenceTransformer

TB_MODEL="aplycaebous/tb-XLM-R-fpt"
BGE_MODEL="nahidstaq/bge-m3-bangla"

print("Checking:",TB_MODEL)
_=AutoConfig.from_pretrained(TB_MODEL)
_=AutoTokenizer.from_pretrained(TB_MODEL)

print("Checking:",BGE_MODEL)
_=SentenceTransformer(BGE_MODEL,device="cpu")

print("✅ checkpoints accessible")


In [ ]:

# 3. Imports + strict LIVE verification
import gc,re,math,json,random,unicodedata,zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
from tqdm.auto import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score,roc_auc_score
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

SEED=2026

LABELS=[
    "is_toxic",
    "implicit_toxicity",
    "sarcasm",
    "contains_profanity",
    "threatening",
    "identity_attack",
    "sexual_content",
]
LI={l:i for i,l in enumerate(LABELS)}
ATTRS=LABELS[1:]

TB_MODEL="aplycaebous/tb-XLM-R-fpt"
BGE_MODEL="nahidstaq/bge-m3-bangla"

MAX_LENGTH=128
BATCH=4
EVAL_BATCH=32
ACCUM=4
EPOCHS=3
SNAPSHOT_EPOCHS=[2,3]

LR=1.2e-5
HEAD_LR=2.5e-5
WEIGHT_DECAY=.01
WARMUP_RATIO=.10

USE_DRIVE_CHECKPOINTS=True

if USE_DRIVE_CHECKPOINTS:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR=Path("/content/drive/MyDrive/V45_FULL_REFIT")
else:
    RUN_DIR=Path("/content/v45_output")

RUN_DIR.mkdir(parents=True,exist_ok=True)

EXPORT_DIR=Path("/content/v45_export")
EXPORT_DIR.mkdir(exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)
assert torch.cuda.is_available(),"Enable T4 GPU."

train=pd.read_csv(DATA_DIR/"train.csv",dtype={"ID":str})
valid=pd.read_csv(DATA_DIR/"validation.csv",dtype={"ID":str})
test=pd.read_csv(DATA_DIR/"test.csv",dtype={"ID":str})
v2v=pd.read_csv(DATA_DIR/"v2_valid.csv",dtype={"ID":str})
v2t=pd.read_csv(DATA_DIR/"v2_test.csv",dtype={"ID":str})
champ=pd.read_csv(DATA_DIR/"submission_v38_champion.csv",dtype={"ID":str})

assert len(train)==3297
assert len(valid)==500
assert len(test)==1000
assert test.ID.tolist()==champ.ID.tolist()
assert test.ID.head(4).tolist()==[
    "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
]

combined=pd.concat([train,valid],ignore_index=True)

def clean_text(x):
    x=unicodedata.normalize("NFKC",str(x))
    x=x.replace("\u200c"," ").replace("\u200d"," ")
    x=re.sub(r"https?://\S+|www\.\S+"," URL ",x)
    x=re.sub(r"@\w+"," USER ",x)
    x=re.sub(r"\s+"," ",x).strip()
    return x

BN_RE=re.compile(r"[\u0980-\u09FF]")
LAT_RE=re.compile(r"[A-Za-z]")

def script_type(x):
    s=str(x)
    bn=bool(BN_RE.search(s))
    lat=bool(LAT_RE.search(s))
    if bn and lat:
        return "mixed"
    if bn:
        return "native"
    return "roman"

for df in [combined,test]:
    df["clean_text"]=df.text.map(clean_text)
    df["script"]=df.clean_text.map(script_type)

print("✅ LIVE split verified")
print("Combined labeled rows:",len(combined))
print("GPU:",torch.cuda.get_device_name(0))


In [ ]:

# 4. Load saved OOF/test features from the proven model family

def load_npz(name):
    return np.load(DATA_DIR/name)

oldb=load_npz("oldb.npz")
oldx=load_npz("oldx.npz")
v17b=load_npz("v17b.npz")
v17x=load_npz("v17x.npz")
v18=load_npz("v18.npz")
v19=load_npz("v19.npz")
v23=load_npz("v23.npz")
v25=load_npz("v25.npz")
v32=load_npz("v32.npz")
v33=load_npz("v33.npz")
v35=load_npz("v35.npz")
v15=load_npz("v15.npz")

fold=np.load(DATA_DIR/"fold_assignment.npy")
assert fold.shape==(3797,)

def logit(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))

def topk(prob,k):
    out=np.zeros(len(prob),int)
    k=int(max(0,min(k,len(prob))))
    if k:
        out[np.argsort(-prob)[:k]]=1
    return out

print("✅ OOF feature family loaded")


In [ ]:

# 5. Reconstruct V14 probability family needed by V38

v14v={}
v14t={}

v14v["is_toxic"]=(
    .875*v2v.is_toxic_prob.to_numpy(float)
    +.125*oldx["valid"][1:3,:,0].mean(0)
)
v14t["is_toxic"]=(
    .875*v2t.is_toxic_prob.to_numpy(float)
    +.125*oldx["test"][1:3,:,0].mean(0)
)

j=LI["implicit_toxicity"]

v14v["implicit_toxicity"]=sigmoid(
    .9*logit(v2v.implicit_toxicity_prob)
    +.1*logit(oldb["valid"].mean(0)[:,j])
)
v14t["implicit_toxicity"]=sigmoid(
    .9*logit(v2t.implicit_toxicity_prob)
    +.1*logit(oldb["test"].mean(0)[:,j])
)

print("✅ V14 toxic/implicit probabilities reconstructed")


In [ ]:

# 6. FULL-DATA BGE threat refit on all 3797 labels

BGE_COMBINED=RUN_DIR/"bge_combined_embeddings.npy"
BGE_TEST=RUN_DIR/"bge_test_embeddings.npy"

if BGE_COMBINED.exists() and BGE_TEST.exists():
    emb=np.load(BGE_COMBINED)
    emb_test=np.load(BGE_TEST)
    print("✅ reuse BGE embeddings")
else:
    model=SentenceTransformer(BGE_MODEL,device="cuda")

    emb=model.encode(
        combined.clean_text.tolist(),
        batch_size=12,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    emb_test=model.encode(
        test.clean_text.tolist(),
        batch_size=12,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    np.save(BGE_COMBINED,emb)
    np.save(BGE_TEST,emb_test)

    del model
    gc.collect()
    torch.cuda.empty_cache()

y_thr=combined["threatening"].to_numpy(int)

sw=np.ones(len(combined),float)
sw[y_thr==1]=3.0

bge_full=LogisticRegression(
    C=.35,
    solver="liblinear",
    max_iter=1500,
    random_state=SEED,
)

bge_full.fit(
    emb,
    y_thr,
    sample_weight=sw
)

bge_full_test=bge_full.predict_proba(
    emb_test
)[:,1]

old_bge_test=v35["bge_test"][:,LI["threatening"]]

# Conservative refit score: average old five-fold BGE family
# with the model trained on all labeled examples.
threat_refit_score=sigmoid(
    .50*logit(old_bge_test)
    +.50*logit(bge_full_test)
)

threat_refit_pred=topk(
    threat_refit_score,
    int(champ["threatening"].sum())
)

print("Threat rows changed vs V38:",
      int((threat_refit_pred!=champ.threatening.to_numpy(int)).sum()))


In [ ]:

# 7. FULL-DATA transliteration-XLM-R fine-tuning

class DS(Dataset):
    def __init__(self,df,tok,labels=True):
        self.x=df.clean_text.astype(str).tolist()
        self.y=df[LABELS].to_numpy(np.float32) if labels else None
        self.tok=tok

    def __len__(self):
        return len(self.x)

    def __getitem__(self,i):
        item=self.tok(
            self.x[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        if self.y is not None:
            item["labels"]=self.y[i]
        return item


class Coll:
    def __init__(self,tok):
        self.tok=tok

    def __call__(self,items):
        y=None
        if "labels" in items[0]:
            y=torch.tensor(
                np.stack([x.pop("labels") for x in items]),
                dtype=torch.float32
            )

        b=self.tok.pad(
            items,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt"
        )

        if y is not None:
            b["labels"]=y

        return b


def competition_loss(logits,y):
    raw=torch.nn.functional.binary_cross_entropy_with_logits(
        logits.float(),
        y,
        reduction="none"
    )

    task=torch.tensor(
        [.50]+[.50/6]*6,
        dtype=torch.float32,
        device=logits.device
    ).unsqueeze(0)

    posboost=torch.tensor(
        [1.0,1.5,1.2,1.2,1.6,1.1,1.5],
        dtype=torch.float32,
        device=logits.device
    ).unsqueeze(0)

    w=torch.where(
        y>.5,
        posboost,
        torch.ones_like(y)
    )

    return (raw*w*task).sum(1).mean()


@torch.no_grad()
def predict(model,loader):
    model.eval()
    out=[]

    for batch in loader:
        batch.pop("labels",None)
        inp={
            k:v.to("cuda",non_blocking=True)
            for k,v in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits=model(**inp).logits

        out.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )

    return np.concatenate(out)


TB_CACHE=RUN_DIR/"tbxlmr_full_test.npy"

if TB_CACHE.exists():
    tb_full_test=np.load(TB_CACHE)
    print("✅ reuse full TB-XLM-R prediction")
else:
    seed_all(SEED)

    tok=AutoTokenizer.from_pretrained(TB_MODEL)
    coll=Coll(tok)

    script_w=combined.script.map({
        "native":.65,
        "mixed":1.60,
        "roman":2.20
    }).to_numpy(float)

    target_w=(
        1.0
        +1.5*combined.implicit_toxicity.to_numpy()
        +.50*combined.sarcasm.to_numpy()
        +1.0*combined.threatening.to_numpy()
        +1.0*combined.sexual_content.to_numpy()
    )

    sampler=WeightedRandomSampler(
        torch.tensor(
            script_w*target_w,
            dtype=torch.double
        ),
        num_samples=len(combined),
        replacement=True
    )

    tl=DataLoader(
        DS(combined,tok,True),
        batch_size=BATCH,
        sampler=sampler,
        num_workers=0,
        pin_memory=True,
        collate_fn=coll
    )

    tel=DataLoader(
        DS(test,tok,False),
        batch_size=EVAL_BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=coll
    )

    model=AutoModelForSequenceClassification.from_pretrained(
        TB_MODEL,
        num_labels=len(LABELS),
        problem_type="multi_label_classification",
        ignore_mismatched_sizes=True,
    ).to("cuda")

    head=[]
    backbone=[]

    for n,p in model.named_parameters():
        if "classifier" in n or "score" in n:
            head.append(p)
        else:
            backbone.append(p)

    opt=torch.optim.AdamW([
        {
            "params":backbone,
            "lr":LR,
            "weight_decay":WEIGHT_DECAY
        },
        {
            "params":head,
            "lr":HEAD_LR,
            "weight_decay":WEIGHT_DECAY
        }
    ])

    steps=math.ceil(len(tl)/ACCUM)*EPOCHS

    sched=get_linear_schedule_with_warmup(
        opt,
        int(steps*WARMUP_RATIO),
        steps
    )

    scaler=torch.amp.GradScaler("cuda")
    snaps=[]

    for epoch in range(1,EPOCHS+1):
        model.train()
        opt.zero_grad(set_to_none=True)

        for step,b in enumerate(
            tqdm(tl,desc=f"Full TB-XLM-R e{epoch}/{EPOCHS}")
        ):
            y=b.pop("labels").to("cuda")

            inp={
                k:v.to("cuda",non_blocking=True)
                for k,v in b.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits=model(**inp).logits

            loss=competition_loss(logits,y)

            scaler.scale(loss/ACCUM).backward()

            if (
                (step+1)%ACCUM==0
                or step+1==len(tl)
            ):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),1.0
                )

                old=scaler.get_scale()
                scaler.step(opt)
                scaler.update()

                if scaler.get_scale()>=old:
                    sched.step()

                opt.zero_grad(set_to_none=True)

        if epoch in SNAPSHOT_EPOCHS:
            snaps.append(
                predict(model,tel)
            )

    tb_full_test=np.mean(snaps,axis=0)
    np.save(TB_CACHE,tb_full_test)

    del model,opt,sched,scaler,tl,tel
    gc.collect()
    torch.cuda.empty_cache()

print("✅ full TB-XLM-R complete")


In [ ]:

# 8. Toxic refit candidate
#
# V38 toxic used:
#   70% V14 + 30% old five-fold TB-XLM-R
# V45 averages old five-fold TB with the new full-data TB model.

old_tb_test=v35["tbxlmr_test"][:,LI["is_toxic"]]

tb_refit_test=sigmoid(
    .50*logit(old_tb_test)
    +.50*logit(tb_full_test[:,LI["is_toxic"]])
)

toxic_refit_score=sigmoid(
    .70*logit(v14t["is_toxic"])
    +.30*logit(tb_refit_test)
)

toxic_refit_pred=topk(
    toxic_refit_score,
    int(champ["is_toxic"].sum())
)

print("Toxic rows changed vs V38:",
      int((toxic_refit_pred!=champ.is_toxic.to_numpy(int)).sum()))


In [ ]:

# 9. Refit implicit meta stacker on all 3797 labeled OOF rows

all_sources={l:[] for l in LABELS}
test_sources={l:[] for l in LABELS}

def add(label,oof_prob,test_prob):
    all_sources[label].append(np.asarray(oof_prob,float))
    test_sources[label].append(np.asarray(test_prob,float))

# Multi-label sources
for oo,tt in [
    (v18["oof"],v18["test"]),
    (v32["stage2_oof"],v32["stage2_test"]),
    (v32["lexical_oof"],v32["lexical_test"]),
    (v33["me5_oof"],v33["me5_test"]),
    (v33["twitter_oof"],v33["twitter_test"]),
    (v35["e5_oof"],v35["e5_test"]),
    (v35["bge_oof"],v35["bge_test"]),
    (v35["tbxlmr_oof"],v35["tbxlmr_test"]),
]:
    for j,l in enumerate(LABELS):
        add(l,oo[:,j],tt[:,j])

# Conditional V23 attributes
for j,l in enumerate(ATTRS):
    add(l,v23["oof"][:,j],v23["test"][:,j])

# V15 rare specialists
for j,l in enumerate([
    "implicit_toxicity",
    "contains_profanity",
    "threatening",
    "sexual_content",
]):
    add(l,v15["oof"][:,j],v15["test"][:,j])

# Toxic context sources
for arr in [v17b,v17x,v19,v25]:
    add("is_toxic",arr["oof"],arr["test"])

add(
    "is_toxic",
    v32["stage1_oof"],
    v32["stage1_test"]
)

TOX_O=list(all_sources["is_toxic"])
TOX_T=list(test_sources["is_toxic"])

def text_meta(df):
    rows=[]
    for s in df.text.astype(str):
        n=max(len(s),1)
        lat=sum(c.isascii() and c.isalpha() for c in s)
        bn=sum("\u0980"<=c<="\u09FF" for c in s)
        words=len(s.split())

        rows.append([
            np.log1p(n),
            np.log1p(words),
            bn/n,
            lat/n,
            np.log1p(s.count("!")),
            np.log1p(s.count("?")),
            int(bool(BN_RE.search(s)) and not bool(LAT_RE.search(s))),
            int(bool(BN_RE.search(s)) and bool(LAT_RE.search(s))),
            int(bool(LAT_RE.search(s)) and not bool(BN_RE.search(s))),
        ])

    return np.asarray(rows,float)

M=text_meta(combined)
MT=text_meta(test)

label="implicit_toxicity"
j=LI[label]

cols=list(all_sources[label])+TOX_O
tcols=list(test_sources[label])+TOX_T

X=np.column_stack(cols)
Xt=np.column_stack(tcols)

X=np.column_stack([
    X,
    logit(X),
    M
])

Xt=np.column_stack([
    Xt,
    logit(Xt),
    MT
])

y=combined[label].to_numpy(int)

# V36 selected C=.01 for implicit.
meta=Pipeline([
    ("scale",StandardScaler()),
    ("lr",LogisticRegression(
        C=.01,
        solver="liblinear",
        max_iter=1200,
        random_state=SEED,
    ))
])

meta.fit(X,y)

implicit_full_meta_test=meta.predict_proba(Xt)[:,1]

# Reconstruct V38 implicit score:
# 60% V14 + 40% old train-only meta.
#
# We cannot recover the exact old continuous meta score from the
# binary V38 CSV, so V45 creates a conservative full-refit rank by
# blending V14 with the newly full-data-fitted meta using the same 60/40 recipe.

implicit_refit_score=sigmoid(
    .60*logit(v14t["implicit_toxicity"])
    +.40*logit(implicit_full_meta_test)
)

implicit_refit_pred=topk(
    implicit_refit_score,
    int(champ["implicit_toxicity"].sum())
)

print("Implicit rows changed vs V38:",
      int((implicit_refit_pred!=champ.implicit_toxicity.to_numpy(int)).sum()))


In [ ]:

# 10. Build four controlled V45 candidates

def make_submission(name, toxic=None, implicit=None, threat=None):
    sub=champ.copy()

    if toxic is not None:
        sub["is_toxic"]=np.asarray(toxic,int)

    if implicit is not None:
        sub["implicit_toxicity"]=np.asarray(implicit,int)

    if threat is not None:
        sub["threatening"]=np.asarray(threat,int)

    assert sub.ID.tolist()==test.ID.tolist()
    assert sub.ID.head(4).tolist()==[
        "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
    ]
    assert sub[LABELS].isin([0,1]).all().all()

    path=EXPORT_DIR/f"submission_v45_{name}.csv"
    sub.to_csv(path,index=False)

    return path,sub

outputs={}

outputs["toxic_refit"]=make_submission(
    "TOXIC_REFIT",
    toxic=toxic_refit_pred
)

outputs["implicit_refit"]=make_submission(
    "IMPLICIT_REFIT",
    implicit=implicit_refit_pred
)

outputs["threat_refit"]=make_submission(
    "THREAT_REFIT",
    threat=threat_refit_pred
)

outputs["combined_refit"]=make_submission(
    "COMBINED_REFIT",
    toxic=toxic_refit_pred,
    implicit=implicit_refit_pred,
    threat=threat_refit_pred
)

print("Created candidates:")
for k,(p,_) in outputs.items():
    print(k,p)


In [ ]:

# 11. Rank-agreement / change audit

rows=[]

for name,(path,sub) in outputs.items():
    rec={"candidate":name}

    for label in [
        "is_toxic",
        "implicit_toxicity",
        "threatening"
    ]:
        rec[f"{label}_changes"]=int(
            (
                sub[label].to_numpy(int)
                !=champ[label].to_numpy(int)
            ).sum()
        )

        rec[f"{label}_positives"]=int(
            sub[label].sum()
        )

    rec["total_changes"]=sum(
        rec[f"{l}_changes"]
        for l in [
            "is_toxic",
            "implicit_toxicity",
            "threatening"
        ]
    )

    rows.append(rec)

audit=pd.DataFrame(rows)
audit.to_csv(
    EXPORT_DIR/"v45_candidate_audit.csv",
    index=False
)

display(audit)

print()
print("IMPORTANT:")
print("V45 uses all 3797 labels for final refit, so there is no untouched")
print("500-row validation score for the refitted models.")
print("Upload v45_results.zip to ChatGPT before any Kaggle submission.")


In [ ]:

# 12. Save refit probabilities + package results

np.savez_compressed(
    EXPORT_DIR/"v45_refit_probabilities.npz",
    bge_full_test=bge_full_test,
    old_bge_test=old_bge_test,
    threat_refit_score=threat_refit_score,
    tb_full_test=tb_full_test,
    old_tb_test=old_tb_test,
    toxic_refit_score=toxic_refit_score,
    implicit_full_meta_test=implicit_full_meta_test,
    implicit_refit_score=implicit_refit_score,
)

summary={
    "protected_public_lb":0.608,
    "minimum_goal":0.6151,
    "main_goal":0.620,
    "combined_labeled_rows":len(combined),
    "strategy":"full-data refit of public-LB-proven V38 components",
    "candidate_files":[
        Path(p).name
        for p,_ in outputs.values()
    ],
    "warning":"Do not submit until ChatGPT reviews v45_results.zip."
}

with open(
    EXPORT_DIR/"v45_summary.json",
    "w"
) as f:
    json.dump(summary,f,indent=2)

zip_path=Path("/content/v45_results.zip")

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in EXPORT_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
